In [1]:
import numpy as np
import pandas as pd
import sklearn.model_selection

In [2]:
data = pd.read_excel("../data/raw/disaster_prediction_dataset.xlsx")
data.head()

,DisNo.,Historic,Classification Key,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,External IDs,Event Name,ISO,...,"Reconstruction Costs, Adjusted ('000 US$)",Insured Damage ('000 US$),"Insured Damage, Adjusted ('000 US$)",Total Damage ('000 US$),"Total Damage, Adjusted ('000 US$)",CPI,Admin Units,GADM Admin Units,Entry Date,Last Update
0,1900-0003-USA,Yes,nat-met-sto-tro,Natural,Meteorological,Storm,Tropical cyclone,NaN,NaN,USA,...,NaN,NaN,NaN,30000.0,1131126.0,2.652223,NaN,NaN,2004-10-18,2023-10-17
1,1900-0006-JAM,Yes,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),NaN,NaN,JAM,...,NaN,NaN,NaN,NaN,NaN,2.652223,NaN,NaN,2003-07-01,2023-09-25
2,1900-0007-JAM,Yes,nat-bio-epi-vir,Natural,Biological,Epidemic,Viral disease,NaN,Gastroenteritis,JAM,...,NaN,NaN,NaN,NaN,NaN,2.652223,NaN,NaN,2003-07-01,2023-09-25
3,1900-0008-JPN,Yes,nat-geo-vol-ash,Natural,Geophysical,Volcanic activity,Ash fall,NaN,NaN,JPN,...,NaN,NaN,NaN,NaN,NaN,2.652223,NaN,NaN,2003-07-01,2023-09-25
4,1900-0009-TUR,Yes,nat-geo-ear-gro,Natural,Geophysical,Earthquake,Ground movement,NaN,NaN,TUR,...,NaN,NaN,NaN,NaN,NaN,2.652223,NaN,NaN,2019-08-05,2023-09-25


In [3]:
data.duplicated().sum()

np.int64(0)

In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 17756 entries, 0 to 17755
Data columns (total 47 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   DisNo.                                     17756 non-null  str    
 1   Historic                                   17756 non-null  str    
 2   Classification Key                         17756 non-null  str    
 3   Disaster Group                             17756 non-null  str    
 4   Disaster Subgroup                          17756 non-null  str    
 5   Disaster Type                              17756 non-null  str    
 6   Disaster Subtype                           17756 non-null  str    
 7   External IDs                               4594 non-null   str    
 8   Event Name                                 4319 non-null   str    
 9   ISO                                        17756 non-null  str    
 10  Country                          

In [6]:
for col in data.columns:
    print(col)

DisNo.
Historic
Classification Key
Disaster Group
Disaster Subgroup
Disaster Type
Disaster Subtype
External IDs
Event Name
ISO
Country
Subregion
Region
Location
Origin
Associated Types
OFDA/BHA Response
Appeal
Declaration
AID Contribution ('000 US$)
Magnitude
Magnitude Scale
Latitude
Longitude
River Basin
Start Year
Start Month
Start Day
End Year
End Month
End Day
Total Deaths
No. Injured
No. Affected
No. Homeless
Total Affected
Reconstruction Costs ('000 US$)
Reconstruction Costs, Adjusted ('000 US$)
Insured Damage ('000 US$)
Insured Damage, Adjusted ('000 US$)
Total Damage ('000 US$)
Total Damage, Adjusted ('000 US$)
CPI
Admin Units
GADM Admin Units
Entry Date
Last Update


In [16]:
drop_cols = [
    # Leakage - human impact
    "Total Deaths", "No. Injured", "No. Affected",
    "No. Homeless", "Total Affected",

    # Leakage - economic damage
    "Reconstruction Costs ('000 US$)",
    "Reconstruction Costs, Adjusted ('000 US$)",
    "Insured Damage ('000 US$)",
    "Insured Damage, Adjusted ('000 US$)",
    "Total Damage ('000 US$)",
    "Total Damage, Adjusted ('000 US$)",

    # Post-event response
    "OFDA/BHA Response", "Appeal", "Declaration",
    "AID Contribution ('000 US$)",

    # Redundant / unnecessary
    "Classification Key", "ISO", "Subregion",
    "Location", "River Basin", "Last Update",

    # Optional removals
    "End Year", "End Month", "End Day",
    "Entry Date" , "Start Day" , "CPI",
]

data = data.drop(columns=drop_cols, errors='ignore')

In [17]:
print(data.columns)

Index(['Historic', 'Disaster Type', 'Country', 'Region', 'Magnitude',
       'Magnitude Scale', 'Latitude', 'Longitude', 'Start Year',
       'Start Month'],
      dtype='str')


In [18]:
data.shape

(17756, 10)

In [19]:
data.columns = data.columns.str.lower().str.replace(" ", "_")

In [20]:
data.columns

Index(['historic', 'disaster_type', 'country', 'region', 'magnitude',
       'magnitude_scale', 'latitude', 'longitude', 'start_year',
       'start_month'],
      dtype='str')

In [21]:
disaster_merge = {
    "Glacial lake outburst flood": "Flood",

    "Mass movement (dry)": "Landslide",
    "Mass movement (wet)": "Landslide",

    "Infestation": "Epidemic",
    "Animal incident": "Epidemic",

    "Impact": "Earthquake",

    "Fog": "Extreme temperature"
}

data["disaster_type"] = data["disaster_type"].replace(disaster_merge)

In [22]:
data["disaster_type"].value_counts()

disaster_type
Flood                  6183
Storm                  5053
Earthquake             1651
Epidemic               1622
Landslide               918
Drought                 803
Extreme temperature     730
Wildfire                514
Volcanic activity       282
Name: count, dtype: int64

In [23]:
data.to_excel("../data/processed/cleaned_disaster_dataset.xlsx", index=False)

In [24]:
data.shape

(17756, 10)